# Fase 0 — Sonda: rate limit, latencia, modelo devuelto, límite de tokens y costo

Objetivo (`plan.md`, Fase 0): confirmar que el proyecto es viable antes de invertir tiempo.

Fuentes leídas antes de escribir este notebook (2026-09-24):
- <https://docs.typesafe.ai/models>: Jev 1.13 (`jev-1.13.0`), $0.042 por Mtok de **entrada** (la salida es gratis), 1.200 req/min, 250.000 tok/s, 64k tokens por request y 32k para `state` + la pregunta más larga.
- <https://docs.typesafe.ai/api>: `POST /v1/systemone`; los errores posibles son 401, 422, 429 y 529.
- <https://docs.typesafe.ai/sdk/python/usage>: `RetryPolicy`, `TypeSafeAPIError`.

Para medir el rate limit real, **los reintentos del SDK están apagados** (`max_retries=0`). Si no, el SDK absorbe los 429 con backoff y no se ven.

Para volver a correrlo: `uv run jupyter nbconvert --to notebook --execute --inplace notebooks/00_probe.ipynb`. Necesita `TYPESAFE_API_KEY` en `.env`.

In [1]:
import asyncio, json, platform, statistics, time
from datetime import datetime, timezone

from dotenv import find_dotenv, load_dotenv
import typesafe_sdk
from typesafe_sdk import (AsyncTypeSafeClient, Choice, Noul, RetryPolicy, Score,
                          TypeSafeAPIError, TypeSafeClient)

load_dotenv(find_dotenv(usecwd=True))

NO_RETRY = RetryPolicy(max_retries=0)
client = TypeSafeClient(retry=NO_RETRY)

RUN = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sdk_version": typesafe_sdk.__version__,
    "python": platform.python_version(),
    "requested_model": "jev-latest (default del SDK)",
}
RUN

{'date_utc': '2026-09-24T13:54:41+00:00',
 'sdk_version': '0.7.0',
 'python': '3.12.13',
 'requested_model': 'jev-latest (default del SDK)'}

## 1. Llamada mínima con las tres primitivas sobre una sentencia SQL

Una sentencia pegada a mano sobre el esquema de Olist. Las preguntas son un borrador del gate de riesgo, no la rúbrica final (esa es de la Fase 4).

In [2]:
SQL = """SELECT c.customer_state, COUNT(*) AS orders, SUM(p.payment_value) AS revenue
FROM orders o, customers c, order_payments p
WHERE o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY revenue DESC;"""

STATE = {"dialect": "postgresql", "sql": SQL}

Q_EFFECT = Choice(
    instructions="What effect would executing `sql` have on the database?",
    criteria={
        "read_only": "Only reads data (SELECT, WITH ... SELECT, EXPLAIN without ANALYZE)",
        "writes_data": "Inserts, updates, deletes or merges rows",
        "changes_schema": "Creates, alters, drops or truncates objects",
        "session_or_admin": "Changes session settings, permissions, or runs admin commands",
    },
)
Q_COST = Score(
    instructions="How expensive would `sql` likely be to run on a production database with millions of rows per table?",
    criteria=[
        "Cheap: filtered by keys or small aggregates",
        "Moderate: full scans of one large table",
        "Expensive: large joins or scans across several big tables",
        "Runaway: unbounded cartesian products or row explosions",
    ],
)
Q_CARTESIAN = Noul(
    instructions="Does `sql` join at least one table without any join condition, producing a cartesian product?",
)
Q_MULTI = Noul(
    instructions="Does `sql` contain more than one SQL statement?",
)
Q_PII = Noul(
    instructions="Does `sql` select columns that identify a person, such as names, emails, phone numbers or street addresses?",
)

Q3 = {"effect": Q_EFFECT, "cost": Q_COST, "cartesian": Q_CARTESIAN}
Q5 = {**Q3, "multi_statement": Q_MULTI, "pii": Q_PII}

t0 = time.perf_counter()
r = client.system_one(STATE, Q3)
first_latency = time.perf_counter() - t0

RUN["returned_model"] = r.model
print("modelo devuelto:", r.model, "| request_id:", r.request_id)
print("usage:", r.usage, f"| latencia: {first_latency:.3f}s")
print(json.dumps({k: v.model_dump() for k, v in r.answers.items()}, indent=2))

modelo devuelto: jev-1.13.0 | request_id: req_01a0d3b26d7575519eb2830e8cdcacec
usage: input_tokens=575 output_tokens=82 | latencia: 0.789s
{
  "effect": {
    "type": "choice",
    "choice": "read_only",
    "confidence": 1.0,
    "probabilities": {
      "changes_schema": 0.0,
      "session_or_admin": 0.0,
      "read_only": 1.0,
      "writes_data": 0.0
    }
  },
  "cost": {
    "type": "score",
    "score": 2.3,
    "confidence": 0.66,
    "legend": {
      "0": "Cheap: filtered by keys or small aggregates",
      "1": "Moderate: full scans of one large table",
      "2": "Expensive: large joins or scans across several big tables",
      "3": "Runaway: unbounded cartesian products or row explosions"
    },
    "probabilities": {
      "0": 0.01,
      "1": 0.01,
      "2": 0.66,
      "3": 0.32
    }
  },
  "cartesian": {
    "type": "noul",
    "noul": 0.82
  }
}


`payments` se une sin condición (`o`–`p`), así que `cartesian` debería salir alto. Ojo: `Noul` no trae `confidence`; la probabilidad **es** la respuesta.

## 2. Latencia: 3 preguntas contra 5 sobre la misma sentencia

Si Jev evalúa las preguntas en paralelo, pasar de 3 a 5 casi no mueve la latencia. Hago llamadas secuenciales, alternando 3 y 5 para que la deriva de la red afecte igual a las dos.

In [3]:
N = 20
lat = {3: [], 5: []}
tokens = {}
for i in range(N):
    for n, qs in ((3, Q3), (5, Q5)):
        t0 = time.perf_counter()
        resp = client.system_one(STATE, qs)
        lat[n].append(time.perf_counter() - t0)
        tokens[n] = resp.usage.input_tokens

def summary(xs):
    xs = sorted(xs)
    return {"median_s": round(statistics.median(xs), 3),
            "p90_s": round(xs[int(0.9 * len(xs)) - 1], 3),
            "min_s": round(xs[0], 3), "max_s": round(xs[-1], 3)}

LATENCY = {n: {**summary(v), "input_tokens": tokens[n]} for n, v in lat.items()}
delta = LATENCY[5]["median_s"] - LATENCY[3]["median_s"]
LATENCY["delta_median_s"] = round(delta, 3)
LATENCY["delta_pct"] = round(100 * delta / LATENCY[3]["median_s"], 1)
LATENCY

{3: {'median_s': 0.287,
  'p90_s': 0.334,
  'min_s': 0.269,
  'max_s': 0.388,
  'input_tokens': 575},
 5: {'median_s': 0.292,
  'p90_s': 0.306,
  'min_s': 0.269,
  'max_s': 0.345,
  'input_tokens': 623},
 'delta_median_s': 0.005,
 'delta_pct': 1.7}

## 3. Los dos límites de tokens del request

Primero calibro cuántos tokens suma cada fila de relleno en `state` y en una pregunta (a partir de `usage.input_tokens`). Después pruebo cada límite de los dos lados:

| caso | `state` + pregunta más larga | `state` + todas las preguntas | se espera |
|---|---|---|---|
| A | ~30k | ~30k | OK |
| B | ~34k | ~34k | rechazo (tope de 32k) |
| C | ~26k | ~56k | OK |
| D | ~26k | ~68k | rechazo (tope de 64k), aunque cada pregunta sola entra en 32k |

In [4]:
ROW = "order 7f3a9c21e4b0 | customer_state SP | status delivered | payment credit_card | value 129.90"

def state_rows(k):
    return {"dialect": "postgresql", "sql": SQL, "result_rows": [ROW] * k}

def long_q(k):
    return Noul(instructions={"reference_rows": [ROW] * k,
                              "question": "Is `sql` consistent with `reference_rows`?"})

def tokens_of(state, qs):
    return client.system_one(state, qs).usage.input_tokens

base = tokens_of(state_rows(0), {"q": Q_CARTESIAN})
per_state_row = (tokens_of(state_rows(200), {"q": Q_CARTESIAN}) - base) / 200
per_q_row = (tokens_of(state_rows(0), {"q": long_q(200)}) - base) / 200
CAL = {"base_tokens": base, "tokens_per_state_row": per_state_row, "tokens_per_question_row": per_q_row}
CAL

{'base_tokens': 361,
 'tokens_per_state_row': 36.995,
 'tokens_per_question_row': 37.02}

In [5]:
def rows_for(tokens, per_row):
    return int(tokens / per_row)

def probe(name, state_tok, q_tok, n_q):
    state = state_rows(rows_for(state_tok, per_state_row))
    if q_tok:
        qs = {f"q{i}": long_q(rows_for(q_tok, per_q_row)) for i in range(n_q)}
    else:
        qs = {"q": Q_CARTESIAN}
    try:
        resp = client.system_one(state, qs, timeout=120)
        return {"case": name, "ok": True, "input_tokens": resp.usage.input_tokens}
    except TypeSafeAPIError as e:
        return {"case": name, "ok": False, "status": e.status, "body": str(e.body)[:300]}

LIMITS = [
    probe("A: ~30k state, 1 pregunta corta", 30_000, 0, 1),
    probe("B: ~34k state, 1 pregunta corta", 34_000, 0, 1),
    probe("C: ~20k state + 6 preguntas de ~6k", 20_000, 6_000, 6),
    probe("D: ~20k state + 8 preguntas de ~6k", 20_000, 6_000, 8),
]
for row in LIMITS:
    print(row)

{'case': 'A: ~30k state, 1 pregunta corta', 'ok': True, 'input_tokens': 30330}
{'case': 'B: ~34k state, 1 pregunta corta', 'ok': False, 'status': 400, 'body': "{'detail': {'error_type': 'max_tokens_exceeded'}}"}
{'case': 'C: ~20k state + 6 preguntas de ~6k', 'ok': True, 'input_tokens': 56463}
{'case': 'D: ~20k state + 8 preguntas de ~6k', 'ok': False, 'status': 400, 'body': "{'detail': {'error_type': 'max_tokens_exceeded'}}"}


## 4. Rate limit real

Mando 1.300 requests chicas (el mismo gate de 3 preguntas) con concurrencia 40 y sin reintentos, y registro el status y el tiempo de cada una. Si el tope de 1.200 req/min se aplica, tiene que aparecer algún 429 antes del final. Costo aproximado: 1.300 × ~400 tokens ≈ 0,5 Mtok ≈ US$0,02.

No mido el tope de 250.000 tok/s: una corrida de eval manda requests de cientos de tokens, así que ese tope no es el que muerde.

In [6]:
TOTAL, CONCURRENCY = 1300, 40

async def burst():
    results = []
    sem = asyncio.Semaphore(CONCURRENCY)
    async with AsyncTypeSafeClient(retry=NO_RETRY) as ac:
        t_start = time.perf_counter()

        async def one(i):
            async with sem:
                t0 = time.perf_counter()
                try:
                    await ac.system_one(STATE, Q3)
                    status = 200
                except TypeSafeAPIError as e:
                    status = e.status
                except Exception as e:  # conexión, timeout
                    status = type(e).__name__
                results.append({"t": time.perf_counter() - t_start, "status": status,
                                "latency": time.perf_counter() - t0})

        await asyncio.gather(*(one(i) for i in range(TOTAL)))
    return results

BURST = await burst()
from collections import Counter
duration = max(r["t"] for r in BURST)
ok = [r for r in BURST if r["status"] == 200]
first_429 = min((r["t"] for r in BURST if r["status"] == 429), default=None)
RATE = {
    "requests": TOTAL, "concurrency": CONCURRENCY,
    "duration_s": round(duration, 1),
    "status_counts": dict(Counter(r["status"] for r in BURST)),
    "achieved_req_per_min": round(len(ok) / duration * 60),
    "ok_in_first_60s": sum(1 for r in ok if r["t"] <= 60),
    "first_429_at_s": None if first_429 is None else round(first_429, 1),
    "ok_before_first_429": None if first_429 is None else sum(1 for r in ok if r["t"] < first_429),
    "median_latency_under_load_s": round(statistics.median(r["latency"] for r in ok), 3),
}
RATE

{'requests': 1300,
 'concurrency': 40,
 'duration_s': 10.9,
 'status_counts': {200: 1300},
 'achieved_req_per_min': 7167,
 'ok_in_first_60s': 1300,
 'first_429_at_s': None,
 'ok_before_first_429': None,
 'median_latency_under_load_s': 0.3}

## 5. Estimación de costo (provisoria)

```
costo ≈ tokens_generador_por_vuelta × vueltas × 40 preguntas × k × configuraciones
      + tokens_juez_LLM × decisiones × k
      + Jev
```

- **Jev** usa el precio verificado ($0,042/Mtok de entrada) y los tokens medidos arriba.
- **Generador y juez:** el plan todavía no fija el modelo, así que el precio por Mtok queda como **placeholder** (`GEN_PRICE_PER_MTOK`) y muestro el costo para varios valores hipotéticos. **No son precios reales de ningún proveedor.** Tokens por vuelta: 4.000 provisorios; se recalculan con el piloto de la Fase 2.
- Volumen de una corrida con Jev (split test): 40 preguntas × 3 vueltas × 2 decisiones (gate + loop) = 240 requests; + 40 sentencias de `statements_test` = 280. Con k = 3 son 840.

In [7]:
JEV_PRICE_PER_MTOK = 0.042          # docs.typesafe.ai/models, 2026-09-24
QUESTIONS, TURNS, K, CONFIGS = 40, 3, 3, 4
GEN_TOKENS_PER_TURN = 4_000         # provisorio
JUDGE_TOKENS_PER_DECISION = 1_500   # provisorio, solo configuración `llm`
DECISIONS_PER_RUN = QUESTIONS * TURNS * 2 + 40
JEV_TOKENS_PER_REQUEST = LATENCY[5]["input_tokens"] * 3  # holgura: estado real con filas

jev_requests = DECISIONS_PER_RUN * K
jev_cost = jev_requests * JEV_TOKENS_PER_REQUEST / 1e6 * JEV_PRICE_PER_MTOK
gen_tokens = GEN_TOKENS_PER_TURN * TURNS * QUESTIONS * K * CONFIGS
judge_tokens = JUDGE_TOKENS_PER_DECISION * DECISIONS_PER_RUN * K

COST = {"jev_requests_test_k3": jev_requests,
        "jev_usd": round(jev_cost, 4),
        "generator_Mtok": gen_tokens / 1e6,
        "judge_Mtok": judge_tokens / 1e6}
for hyp_price in (1, 3, 15):  # US$/Mtok HIPOTÉTICO, no es el precio de un proveedor
    COST[f"total_usd_if_llm_{hyp_price}_per_Mtok"] = round(
        (gen_tokens + judge_tokens) / 1e6 * hyp_price + jev_cost, 2)
COST

{'jev_requests_test_k3': 840,
 'jev_usd': 0.0659,
 'generator_Mtok': 5.76,
 'judge_Mtok': 1.26,
 'total_usd_if_llm_1_per_Mtok': 7.09,
 'total_usd_if_llm_3_per_Mtok': 21.13,
 'total_usd_if_llm_15_per_Mtok': 105.37}

## 6. Criterio de salida 🚦

> El rate limit permite completar una corrida de eval sin throttling.

El pico que necesita una corrida: el agente es secuencial por pregunta (una vuelta del generador tarda segundos), así que con `P` preguntas en paralelo el ritmo de Jev es ≈ `P × 2 decisiones / segundos_por_vuelta × 60` req/min.

In [8]:
SECONDS_PER_TURN = 3   # optimista (generador rápido); más lento = menos presión
PARALLEL_QUESTIONS = 10
need_per_min = PARALLEL_QUESTIONS * 2 / SECONDS_PER_TURN * 60
# Conservador: el menor entre lo publicado y lo observado. Los docs avisan que el
# límite cambia sin aviso, así que no se planifica contra el pico observado.
observed = RATE["ok_before_first_429"] if RATE["first_429_at_s"] is not None else RATE["ok_in_first_60s"]
capacity = min(1_200, observed)
GATE = {"need_req_per_min": round(need_per_min),
        "measured_capacity_req_per_min": capacity,
        "margin_x": round(capacity / need_per_min, 1),
        "pass": capacity >= need_per_min}
print(json.dumps({"RUN": RUN, "LATENCY": LATENCY, "CAL": CAL, "LIMITS": LIMITS,
                  "RATE": RATE, "COST": COST, "GATE": GATE}, indent=2, default=str))

{
  "RUN": {
    "date_utc": "2026-09-24T13:54:41+00:00",
    "sdk_version": "0.7.0",
    "python": "3.12.13",
    "requested_model": "jev-latest (default del SDK)",
    "returned_model": "jev-1.13.0"
  },
  "LATENCY": {
    "3": {
      "median_s": 0.287,
      "p90_s": 0.334,
      "min_s": 0.269,
      "max_s": 0.388,
      "input_tokens": 575
    },
    "5": {
      "median_s": 0.292,
      "p90_s": 0.306,
      "min_s": 0.269,
      "max_s": 0.345,
      "input_tokens": 623
    },
    "delta_median_s": 0.005,
    "delta_pct": 1.7
  },
  "CAL": {
    "base_tokens": 361,
    "tokens_per_state_row": 36.995,
    "tokens_per_question_row": 37.02
  },
  "LIMITS": [
    {
      "case": "A: ~30k state, 1 pregunta corta",
      "ok": true,
      "input_tokens": 30330
    },
    {
      "case": "B: ~34k state, 1 pregunta corta",
      "ok": false,
      "status": 400,
      "body": "{'detail': {'error_type': 'max_tokens_exceeded'}}"
    },
    {
      "case": "C: ~20k state + 6 preguntas de

## Resultados (corrida del 2026-09-24)

| medición | valor |
|---|---|
| modelo devuelto | `jev-1.13.0` (se pidió `jev-latest`) |
| latencia, 3 preguntas (mediana / p90) | 0,287 s / 0,334 s |
| latencia, 5 preguntas (mediana / p90) | 0,292 s / 0,306 s |
| diferencia 5 contra 3 | +5 ms (+1,7%): **paralelismo confirmado** |
| primera llamada (en frío, incluye TLS) | 0,79 s |
| tope `state` + pregunta más larga | 30k OK, 34k rechazado: **coincide con 32k** |
| tope `state` + todas las preguntas | 56k OK, 68k rechazado: **coincide con 64k** |
| error por exceso de tokens | **HTTP 400** `max_tokens_exceeded` (los docs de la API listan 422 para validación; ver `NOTES.md`) |
| rate limit | 1.300/1.300 OK en 10,9 s (~7.200 req/min de ritmo), **ningún 429**: no se alcanzó el tope |
| latencia bajo carga (concurrencia 40) | mediana 0,300 s |
| Jev para test con k = 3 | 840 requests ≈ US$0,07 |
| generador + juez | 7,0 Mtok; el costo depende del modelo que se elija |

**🚦 Criterio de salida: CUMPLIDO.** Con 10 preguntas en paralelo y 3 s por vuelta, una corrida necesita ~400 req/min. Aun tomando el tope publicado (1.200 req/min) y no el observado, hay 3× de margen. No hace falta achicar los datasets.

**Pendiente:** recalcular el costo con los tokens reales del piloto de la Fase 2 cuando esté fijado el modelo generador. La latencia se midió desde la máquina de desarrollo (Buenos Aires).